
<span style="font-size: 32px; font-weight: bold; color: #003366;">Proyecto Ausentismo en la administración pública de una provincia argentina</span>



El estudio de las pautas de ausentismo en las organizaciones públicas constituye un campo crítico dentro del análisis organizacional y la economía del trabajo. La disponibilidad de grandes volúmenes de datos transaccionales permite ir más allá del análisis estadístico descriptivo e incursionar en la modelación predictiva del comportamiento laboral.

Este trabajo final intenta abordar el ciclo completo de un proyecto de Data Science aplicado a la función pública: 
<p style="font-size: 16px; line-height: 1.6;">
    <mark style="background-color: #e6f2ff; color: #003366; padding: 2px 5px; border-radius: 3px;">
    Desde la ingesta, limpieza e ingeniería de variables sobre datos reales de ausentismo, 
    hasta el entrenamiento y evaluación de algoritmos de clasificación, regresión y análisis de supervivencia.
    </mark>
</p>

 Los resultados obtenidos ofrecen una metodología replicable para predecir eventos de inasistencia y respaldar la toma de decisiones basada en datos dentro del sector público.


In [3]:
import pandas as pd
import numpy as np

# Cargar el dataset con codificación latin1 y separador ';'
file_path = 'prueba2025_b.csv'
df_ausentismo = pd.read_csv(file_path, encoding='latin1', sep=';')

print(f"Dataset cargado correctamente: {df.shape[0]:,} filas y {df.shape[1]} columnas.")
df.head()

Dataset cargado correctamente: 486,002 filas y 15 columnas.


,Nro legajo,Sublegajo,Escalafon,Cod Art ausencia,Descripcion articulo ausencia,Es de Salud?,Fecha Desde,Fecha Hasta,Cantidad dias,Cant Dias Permitido Mes,Cantidad Dias Permitido Año,Dependencia,Fecha Nacimiento,Genero,Antigüedad
0,283821,30,3,1,10A-6IIIA ENF.COMUN,yes,2025-04-15,2025-04-16,2,0,45,138,1980-08-14,F,5
1,283821,30,3,1,10A-6IIIA ENF.COMUN,yes,2025-04-28,2025-04-30,3,0,45,138,1980-08-14,F,5
2,283821,30,3,1,10A-6IIIA ENF.COMUN,yes,2025-08-12,2025-08-14,3,0,45,138,1980-08-14,F,5
3,283821,30,3,1,10A-6IIIA ENF.COMUN,yes,2025-10-20,2025-10-20,1,0,45,138,1980-08-14,F,5
4,283821,30,3,1,10A-6IIIA ENF.COMUN,yes,2025-11-26,2025-11-26,1,0,45,138,1980-08-14,F,5


In [4]:
# Convertir columnas de fechas al formato datetime de Pandas
df_ausentismo['Fecha Desde'] = pd.to_datetime(df['Fecha Desde'], format='%d/%m/%Y', errors='coerce')
df_ausentismo['Fecha Hasta'] = pd.to_datetime(df['Fecha Hasta'], format='%d/%m/%Y', errors='coerce')
df_ausentismo['Fecha Nacimiento'] = pd.to_datetime(df['Fecha Nacimiento'], format='%d/%m/%Y', errors='coerce')

# Ordenar cronológicamente por legajo y fecha para garantizar cálculos de secuencias temporales correctos
df_ausentismo_ord = df_ausentismo.sort_values(['Nro legajo', 'Fecha Desde']).reset_index(drop=True)

print("Conversión de fechas completada.")

Conversión de fechas completada.


In [5]:
# 1. Edad del agente al momento del inicio de la licencia
df_ausentismo_ord['Edad_Agente'] = (df_ausentismo_ord['Fecha Desde'] - df_ausentismo_ord['Fecha Nacimiento']).dt.days // 365

# 2. Extracción de componentes estacionales
df_ausentismo_ord['Mes_Inicio'] = df_ausentismo_ord['Fecha Desde'].dt.month
df_ausentismo_ord['Trimestre'] = df_ausentismo_ord['Fecha Desde'].dt.quarter
df_ausentismo_ord['Dia_Semana_Inicio'] = df_ausentismo_ord['Fecha Desde'].dt.day_name()

# 3. Indicador de día adyacente a fin de semana (Lunes, Viernes, Sábado o Domingo)
df_ausentismo_ord['Es_Fin_De_Semana_Adyacente'] = df_ausentismo_ord['Fecha Desde'].dt.dayofweek.isin([0, 4, 5, 6]).astype(int)

df_ausentismo_ord[['Fecha Desde', 'Edad_Agente', 'Mes_Inicio', 'Trimestre', 'Dia_Semana_Inicio', 'Es_Fin_De_Semana_Adyacente']].head()

,Fecha Desde,Edad_Agente,Mes_Inicio,Trimestre,Dia_Semana_Inicio,Es_Fin_De_Semana_Adyacente
0,2025-04-15,44.0,4,2,Tuesday,0
1,2025-04-28,44.0,4,2,Monday,1
2,2025-08-12,45.0,8,3,Tuesday,0
3,2025-10-20,45.0,10,4,Monday,1
4,2025-11-26,45.0,11,4,Wednesday,0


In [6]:
# Transformar la variable de salud a numérico (1: Sí, 0: No)
df_ausentismo_ord['Es_Salud_Num'] = (df_ausentismo_ord['Es de Salud?'] == 'yes').astype(int)

# 1. Cantidad de días transcurridos desde la última licencia registrada para el mismo legajo
df_ausentismo_ord['Dias_Desde_Ultima_Ausencia'] = df_ausentismo_ord.groupby('Nro legajo')['Fecha Desde'].diff().dt.days

# Fillna para la primera licencia del agente
df_ausentismo_ord['Dias_Desde_Ultima_Ausencia'] = df_ausentismo_ord['Dias_Desde_Ultima_Ausencia'].fillna(-1)

# 2. Total acumulado de licencias tomadas previamente por el mismo agente en el año
df_ausentismo_ord['Num_Licencias_Previas_Agente'] = df_ausentismo_ord.groupby('Nro legajo').cumcount()

# 3. Total acumulado de días de ausencia en el año previo a la fecha actual
df_ausentismo_ord['Dias_Ausencia_Acumulados_Año'] = df_ausentismo_ord.groupby('Nro legajo')['Cantidad dias'].cumsum() - df['Cantidad dias']

# 4. Tasa de consumo del cupo anual permitido
# Se evita la división por cero asignando 0 si Cantidad Dias Permitido Año es 0
df_ausentismo_ord['Tasa_Uso_Cupo_Anual'] = np.where(
    df_ausentismo_ord['Cantidad Dias Permitido Año'] > 0,
    df_ausentismo_ord['Dias_Ausencia_Acumulados_Año'] / df_ausentismo_ord['Cantidad Dias Permitido Año'],
    0
)

df_ausentismo_ord[['Nro legajo', 'Fecha Desde', 'Num_Licencias_Previas_Agente', 'Dias_Ausencia_Acumulados_Año', 'Tasa_Uso_Cupo_Anual']].head()

,Nro legajo,Fecha Desde,Num_Licencias_Previas_Agente,Dias_Ausencia_Acumulados_Año,Tasa_Uso_Cupo_Anual
0,283821,2025-04-15,0,0,0.000000
1,283821,2025-04-28,1,2,0.044444
2,283821,2025-08-12,2,5,0.111111
3,283821,2025-10-20,3,8,0.177778
4,283821,2025-11-26,4,9,0.200000
